In [7]:
"""
MODULE 3 - PRACTICAL DEMONSTRATION
Healthcare Accessibility Intelligence System

A complete data-aware agent that autonomously analyzes healthcare accessibility
in any city worldwide, demonstrating all Module 3 concepts:
- Autonomous data discovery
- Multi-source integration  
- Spatial reasoning
- Quality assessment
- Validation frameworks

Author: PixelGEO
Module: 3 - Building Data-Aware Agents
"""

import geopandas as gpd
import pandas as pd
import folium
from shapely.geometry import Point, Polygon
import requests
import json
from datetime import datetime
import logging
from typing import Dict, List, Tuple, Optional
import time

In [8]:
# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


class HealthcareAccessibilityAgent:
    """
    Data-aware agent for healthcare accessibility analysis
    
    Demonstrates:
    - Autonomous data discovery from OpenStreetMap
    - Multi-source integration
    - Spatial reasoning and distance calculations
    - Quality assessment framework
    - Validation and error handling
    """
    
    def __init__(self):
        """Initialize the agent with data source registry"""
        self.cache = {}
        self.data_sources = self._initialize_source_registry()
        self.quality_thresholds = {
            'min_facilities': 5,
            'min_population_points': 10,
            'max_distance_km': 50
        }
        
    def _initialize_source_registry(self) -> Dict:
        """
        Initialize the data source registry
        Demonstrates: Source knowledge pattern
        """
        return {
            'osm': {
                'base_url': 'https://overpass-api.de/api/interpreter',
                'features': ['healthcare', 'buildings', 'roads'],
                'coverage': 'global',
                'cost': 'free'
            },
            'nominatim': {
                'base_url': 'https://nominatim.openstreetmap.org/search',
                'features': ['geocoding'],
                'coverage': 'global',
                'cost': 'free'
            }
            # 'nhs_portal': {
            #     'base_url': 'https://api.nhs.uk/hospitals',
            #     'features': ['healthcare', 'waiting_times'],
            #     'coverage': 'UK only',
            #     'cost': 'free',
            #     'requires_auth': False}
        }
    
    def analyze_healthcare_accessibility(
        self, 
        location: str, 
        radius_km: float = 10
    ) -> Dict:
        """
        Main analysis function - completely autonomous
        
        Args:
            location: City or place name
            radius_km: Analysis radius in kilometres
            
        Returns:
            Complete analysis results with confidence scores
        """
        logger.info(f"=== Starting Healthcare Accessibility Analysis ===")
        logger.info(f"Location: {location}")
        logger.info(f"Radius: {radius_km} km")
        
        try:
            # Step 1: Geocode location
            logger.info("\n[STEP 1] Geocoding location...")
            lat, lon = self._geocode_location(location)
            logger.info(f"✓ Coordinates: {lat:.4f}, {lon:.4f}")
            
            # Step 2: Discover healthcare facilities
            logger.info("\n[STEP 2] Discovering healthcare facilities...")
            facilities = self._discover_healthcare_facilities(
                location, lat, lon, radius_km
            )
            logger.info(f"✓ Found {len(facilities)} healthcare facilities")
            
            if len(facilities) == 0:
                return self._empty_result_handler(location, "No facilities found")
            
            # Step 3: Discover population centers
            logger.info("\n[STEP 3] Discovering population centers...")
            population = self._discover_population_centers(
                location, lat, lon, radius_km
            )
            logger.info(f"✓ Found {len(population)} population points")
            
            if len(population) == 0:
                return self._empty_result_handler(location, "No population data")
            
            # Step 4: Validate data quality
            logger.info("\n[STEP 4] Validating data quality...")
            validation = self._validate_data_quality(facilities, population)
            logger.info(f"✓ Validation: {validation['status']}")
            logger.info(f"  Confidence: {validation['confidence']}")
            
            # Step 5: Calculate accessibility
            logger.info("\n[STEP 5] Calculating accessibility scores...")
            accessibility = self._calculate_accessibility_scores(
                facilities, population
            )
            logger.info(f"✓ Analyzed {len(accessibility)} areas")
            
            # Step 6: Generate insights
            logger.info("\n[STEP 6] Generating insights...")
            insights = self._generate_insights(accessibility, validation)
            logger.info(f"✓ Generated {len(insights['recommendations'])} recommendations")
            
            # Step 7: Create visualization
            logger.info("\n[STEP 7] Creating visualization...")
            map_obj = self._create_visualization(
                location, lat, lon, facilities, accessibility
            )
            map_filename = f"healthcare_accessibility_{location.replace(' ', '_').lower()}.html"
            map_obj.save(map_filename)
            logger.info(f"✓ Map saved: {map_filename}")
            
            logger.info("\n=== Analysis Complete ===\n")
            
            return {
                'success': True,
                'location': location,
                'coordinates': (lat, lon),
                'facilities_found': len(facilities),
                'areas_analyzed': len(accessibility),
                'validation': validation,
                'insights': insights,
                'map_file': map_filename,
                'data': {
                    'facilities': facilities,
                    'accessibility': accessibility
                }
            }
            
        except Exception as e:
            logger.error(f"Analysis failed: {str(e)}")
            return {
                'success': False,
                'error': str(e),
                'recommendation': 'Check logs for details'
            }
    
    def _geocode_location(self, location: str) -> Tuple[float, float]:
        """
        Convert location name to coordinates
        Demonstrates: Intent-based discovery
        """
        # Check cache first
        if location in self.cache:
            logger.info("  Using cached coordinates")
            return self.cache[location]
        
        # Query Nominatim
        url = self.data_sources['nominatim']['base_url']
        params = {
            'q': location,
            'format': 'json',
            'limit': 1
        }
        headers = {'User-Agent': 'HealthcareAccessibilityAgent/1.0'}
        
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)
            response.raise_for_status()
            
            if response.json():
                result = response.json()[0]
                lat, lon = float(result['lat']), float(result['lon'])
                
                # Validate coordinates
                if not (-90 <= lat <= 90 and -180 <= lon <= 180):
                    raise ValueError(f"Invalid coordinates: {lat}, {lon}")
                
                # Cache result
                self.cache[location] = (lat, lon)
                return lat, lon
            else:
                raise ValueError(f"Location not found: {location}")
                
        except requests.RequestException as e:
            raise ConnectionError(f"Geocoding failed: {e}")
    
    def _discover_healthcare_facilities(
        self,
        location: str,
        lat: float,
        lon: float,
        radius_km: float
    ) -> List[Dict]:
        """
        Discover healthcare facilities from OpenStreetMap
        Demonstrates: OSM Overpass API querying, data discovery
        """
        radius_m = radius_km * 1000
        
        # Construct Overpass query
        query = f"""
        [out:json][timeout:25];
        (
          node["amenity"="hospital"](around:{radius_m},{lat},{lon});
          way["amenity"="hospital"](around:{radius_m},{lat},{lon});
          node["amenity"="clinic"](around:{radius_m},{lat},{lon});
          way["amenity"="clinic"](around:{radius_m},{lat},{lon});
          node["amenity"="doctors"](around:{radius_m},{lat},{lon});
          way["amenity"="doctors"](around:{radius_m},{lat},{lon});
          node["healthcare"](around:{radius_m},{lat},{lon});
          way["healthcare"](around:{radius_m},{lat},{lon});
        );
        out center;
        """
        
        logger.info(f"  Querying OSM Overpass API...")
        logger.info(f"  Search radius: {radius_km} km ({radius_m} m)")
        
        try:
            response = requests.post(
                self.data_sources['osm']['base_url'],
                data={'data': query},
                timeout=30
            )
            response.raise_for_status()
            
            # Parse response
            facilities = self._parse_osm_response(response.json())
            
            logger.info(f"  Retrieved {len(facilities)} facilities from OSM")
            
            return facilities
            
        except requests.RequestException as e:
            logger.warning(f"  OSM query failed: {e}")
            return self._fallback_facilities(location)
    
    def _parse_osm_response(self, data: Dict) -> List[Dict]:
        """
        Parse OSM API response into structured format
        Demonstrates: Data integration, schema alignment
        """
        facilities = []
        
        for element in data.get('elements', []):
            # Get coordinates
            if 'center' in element:
                lat, lon = element['center']['lat'], element['center']['lon']
            elif 'lat' in element and 'lon' in element:
                lat, lon = element['lat'], element['lon']
            else:
                continue
            
            # Validate coordinates
            if not (-90 <= lat <= 90 and -180 <= lon <= 180):
                logger.warning(f"  Invalid coordinates skipped: {lat}, {lon}")
                continue
            
            # Extract tags
            tags = element.get('tags', {})
            facility_type = (
                tags.get('amenity') or 
                tags.get('healthcare') or 
                'unknown'
            )
            
            facilities.append({
                'id': element.get('id'),
                'type': facility_type,
                'name': tags.get('name', 'Unnamed Facility'),
                'lat': lat,
                'lon': lon,
                'tags': tags,
                'source': 'OpenStreetMap'
            })
        
        return facilities
    
    def _discover_population_centers(
        self,
        location: str,
        lat: float,
        lon: float,
        radius_km: float
    ) -> List[Dict]:
        """
        Discover population centers from residential buildings
        Demonstrates: Proxy data usage, spatial reasoning
        """
        radius_m = radius_km * 1000
        
        # Query for residential buildings
        query = f"""
        [out:json][timeout:25];
        (
          way["building"="residential"](around:{radius_m},{lat},{lon});
          way["building"="apartments"](around:{radius_m},{lat},{lon});
          way["building"="house"](around:{radius_m},{lat},{lon});
          way["landuse"="residential"](around:{radius_m},{lat},{lon});
        );
        out center;
        """
        
        logger.info(f"  Querying OSM for residential areas...")
        
        try:
            response = requests.post(
                self.data_sources['osm']['base_url'],
                data={'data': query},
                timeout=30
            )
            response.raise_for_status()
            
            # Parse and create population points
            population = []
            for element in response.json().get('elements', []):
                if 'center' in element:
                    center = element['center']
                    population.append({
                        'lat': center['lat'],
                        'lon': center['lon'],
                        'type': 'residential',
                        'source': 'OpenStreetMap'
                    })
            
            logger.info(f"  Retrieved {len(population)} residential points from OSM")
            
            # If sparse, generate grid points
            if len(population) < self.quality_thresholds['min_population_points']:
                logger.info(f"  Sparse data detected, generating grid points...")
                population = self._generate_population_grid(lat, lon, radius_km)
            
            return population
            
        except requests.RequestException as e:
            logger.warning(f"  Population query failed: {e}")
            return self._generate_population_grid(lat, lon, radius_km)
    
    def _generate_population_grid(
        self, 
        lat: float, 
        lon: float, 
        radius_km: float
    ) -> List[Dict]:
        """
        Generate grid of population points as fallback
        Demonstrates: Fallback strategy, synthetic data
        """
        grid_size = 0.01  # Roughly 1 km
        points_per_side = int(radius_km / 1.1)  # Points within radius
        
        population = []
        for i in range(-points_per_side, points_per_side + 1):
            for j in range(-points_per_side, points_per_side + 1):
                point_lat = lat + (i * grid_size)
                point_lon = lon + (j * grid_size)
                
                # Check if within radius (rough check)
                distance = ((point_lat - lat)**2 + (point_lon - lon)**2)**0.5
                if distance * 111 <= radius_km:  # Rough conversion to km
                    population.append({
                        'lat': point_lat,
                        'lon': point_lon,
                        'type': 'estimated',
                        'source': 'Generated Grid'
                    })
        
        logger.info(f"  Generated {len(population)} grid points")
        return population
    
    def _validate_data_quality(
        self,
        facilities: List[Dict],
        population: List[Dict]
    ) -> Dict:
        """
        Validate data quality across multiple dimensions
        Demonstrates: Quality assessment framework
        """
        issues = []
        scores = {}
        
        # 1. Completeness check
        if len(facilities) < self.quality_thresholds['min_facilities']:
            issues.append(f"Few facilities found: {len(facilities)}")
            scores['completeness'] = len(facilities) / self.quality_thresholds['min_facilities']
        else:
            scores['completeness'] = 1.0
        
        # 2. Coordinate validation
        valid_coords = all(
            -90 <= f['lat'] <= 90 and -180 <= f['lon'] <= 180 
            for f in facilities
        )
        scores['accuracy'] = 1.0 if valid_coords else 0.0
        if not valid_coords:
            issues.append("Invalid coordinates detected")
        
        # 3. Coverage assessment
        synthetic_pop = sum(1 for p in population if p.get('source') == 'Generated Grid')
        if synthetic_pop > 0:
            issues.append(f"Using {synthetic_pop} estimated population points")
            scores['coverage'] = 0.6
        else:
            scores['coverage'] = 1.0
        
        # 4. Data source quality
        osm_count = sum(1 for f in facilities if f.get('source') == 'OpenStreetMap')
        scores['source_quality'] = osm_count / len(facilities) if facilities else 0.0
        
        # Calculate overall score
        overall_score = sum(scores.values()) / len(scores)
        
        # Determine confidence
        if overall_score >= 0.8:
            confidence = "HIGH"
            status = "GOOD"
        elif overall_score >= 0.6:
            confidence = "MEDIUM"
            status = "ACCEPTABLE"
        else:
            confidence = "LOW"
            status = "LIMITED"
        
        return {
            'status': status,
            'confidence': confidence,
            'overall_score': overall_score,
            'dimension_scores': scores,
            'issues': issues
        }
    
    def _calculate_accessibility_scores(
        self,
        facilities: List[Dict],
        population: List[Dict]
    ) -> List[Dict]:
        """
        Calculate accessibility scores for each population point
        Demonstrates: Spatial reasoning, distance calculations
        """
        accessibility_data = []
        
        for pop_point in population:
            pop_lat, pop_lon = pop_point['lat'], pop_point['lon']
            
            # Calculate distances to all facilities
            distances = []
            for facility in facilities:
                # Haversine distance calculation (geodesic)
                distance_km = self._haversine_distance(
                    pop_lat, pop_lon,
                    facility['lat'], facility['lon']
                )
                
                distances.append({
                    'facility': facility,
                    'distance_km': distance_km
                })
            
            # Sort by distance
            distances.sort(key=lambda x: x['distance_km'])
            nearest_three = distances[:3]
            
            # Calculate accessibility score
            # 100 = healthcare within 1km
            # 50 = healthcare within 5km  
            # 0 = healthcare beyond 10km
            if len(nearest_three) > 0:
                nearest_dist = nearest_three[0]['distance_km']
                
                if nearest_dist <= 1:
                    score = 100
                elif nearest_dist <= 5:
                    # Linear interpolation between 100 and 50
                    score = 100 - ((nearest_dist - 1) / 4) * 50
                elif nearest_dist <= 10:
                    # Linear interpolation between 50 and 0
                    score = 50 - ((nearest_dist - 5) / 5) * 50
                else:
                    score = 0
            else:
                score = 0
            
            # Count facilities within 5km
            facilities_5km = len([d for d in distances if d['distance_km'] <= 5])
            
            accessibility_data.append({
                'location': pop_point,
                'score': score,
                'nearest_facilities': nearest_three,
                'facility_count_5km': facilities_5km,
                'nearest_distance_km': nearest_three[0]['distance_km'] if nearest_three else None
            })
        
        return accessibility_data
    
    def _haversine_distance(
        self,
        lat1: float,
        lon1: float,
        lat2: float,
        lon2: float
    ) -> float:
        """
        Calculate geodesic distance using Haversine formula
        Demonstrates: Proper distance calculation
        """
        from math import radians, sin, cos, sqrt, atan2
        
        R = 6371  # Earth's radius in kilometres
        
        lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
        
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        c = 2 * atan2(sqrt(a), sqrt(1-a))
        
        distance = R * c
        return distance
    
    def _generate_insights(
        self,
        accessibility: List[Dict],
        validation: Dict
    ) -> Dict:
        """
        Generate human-readable insights and recommendations
        Demonstrates: Insight generation, decision support
        """
        scores = [a['score'] for a in accessibility]
        distances = [a['nearest_distance_km'] for a in accessibility if a['nearest_distance_km']]
        
        insights = {
            'average_score': sum(scores) / len(scores),
            'min_score': min(scores),
            'max_score': max(scores),
            'average_distance_km': sum(distances) / len(distances) if distances else None,
            'underserved_areas': len([s for s in scores if s < 50]),
            'well_served_areas': len([s for s in scores if s >= 80]),
            'total_analyzed': len(scores),
            'data_confidence': validation['confidence']
        }
        
        # Generate recommendations
        recommendations = []
        
        if insights['average_score'] < 60:
            recommendations.append(
                "Overall healthcare accessibility is below optimal levels. "
                "Consider strategic placement of new facilities."
            )
        
        if insights['underserved_areas'] > len(scores) * 0.3:
            recommendations.append(
                f"{insights['underserved_areas']} areas are significantly underserved "
                f"(score < 50). Priority areas for healthcare infrastructure development."
            )
        
        if insights['average_distance_km'] and insights['average_distance_km'] > 5:
            recommendations.append(
                f"Average distance to nearest facility is {insights['average_distance_km']:.1f}km. "
                "Consider mobile health units or satellite clinics."
            )
        
        if validation['confidence'] in ['MEDIUM', 'LOW']:
            recommendations.append(
                f"Data confidence is {validation['confidence']}. "
                "Results should be validated with local data sources."
            )
        
        insights['recommendations'] = recommendations
        
        return insights
    
    def _create_visualization(
        self,
        location: str,
        lat: float,
        lon: float,
        facilities: List[Dict],
        accessibility: List[Dict]
    ) -> folium.Map:
        """
        Create interactive map visualization
        Demonstrates: Results presentation
        """
        # Create base map
        m = folium.Map(location=[lat, lon], zoom_start=12)
        
        # Add healthcare facilities
        for facility in facilities:
            folium.Marker(
                location=[facility['lat'], facility['lon']],
                popup=(
                    f"<b>{facility['name']}</b><br>"
                    f"Type: {facility['type']}<br>"
                    f"Source: {facility['source']}"
                ),
                icon=folium.Icon(color='red', icon='plus', prefix='fa'),
                tooltip=facility['name']
            ).add_to(m)
        
        # Add population centers with accessibility scores
        for data in accessibility:
            pop = data['location']
            score = data['score']
            
            # Color based on score
            if score >= 80:
                color = 'green'
                fill_color = '#4caf50'
            elif score >= 50:
                color = 'orange'
                fill_color = '#ff9800'
            else:
                color = 'red'
                fill_color = '#f44336'
            
            # Create popup with details
            nearest = data['nearest_facilities'][0] if data['nearest_facilities'] else None
            popup_html = f"""
            <div style='width: 200px'>
                <h4>Accessibility Score: {score:.0f}/100</h4>
                <p><b>Facilities within 5km:</b> {data['facility_count_5km']}</p>
            """
            if nearest:
                popup_html += f"""
                <p><b>Nearest Facility:</b><br>
                {nearest['facility']['name']}<br>
                Distance: {nearest['distance_km']:.2f} km</p>
                """
            popup_html += "</div>"
            
            folium.CircleMarker(
                location=[pop['lat'], pop['lon']],
                radius=6,
                popup=folium.Popup(popup_html, max_width=250),
                color=color,
                fill=True,
                fillColor=fill_color,
                fillOpacity=0.7,
                weight=2
            ).add_to(m)
        
        # Add title
        title_html = f'''
        <div style="position: fixed; 
                    top: 10px; left: 50px; width: 400px; height: 90px; 
                    background-color: white; border:2px solid grey; z-index:9999; 
                    font-size:14px; padding: 10px">
        <h4 style="margin:0">Healthcare Accessibility Analysis</h4>
        <p style="margin:5px 0"><b>Location:</b> {location}</p>
        <p style="margin:5px 0"><b>Facilities:</b> {len(facilities)} | 
           <b>Areas Analyzed:</b> {len(accessibility)}</p>
        </div>
        '''
        m.get_root().html.add_child(folium.Element(title_html))
        
        # Add legend
        legend_html = '''
        <div style="position: fixed; 
                    bottom: 50px; right: 50px; width: 200px; height: 160px; 
                    background-color: white; border:2px solid grey; z-index:9999; 
                    font-size:14px; padding: 10px">
        <p style="margin:0"><b>Healthcare Accessibility</b></p>
        <p style="margin:5px 0"><i class="fa fa-circle" style="color:green"></i> High (80-100)</p>
        <p style="margin:5px 0"><i class="fa fa-circle" style="color:orange"></i> Medium (50-79)</p>
        <p style="margin:5px 0"><i class="fa fa-circle" style="color:red"></i> Low (0-49)</p>
        <p style="margin:5px 0"><i class="fa fa-plus" style="color:red"></i> Healthcare Facility</p>
        </div>
        '''
        m.get_root().html.add_child(folium.Element(legend_html))
        
        return m
    
    def _fallback_facilities(self, location: str) -> List[Dict]:
        """Fallback when OSM query fails"""
        logger.warning("  Using empty fallback for facilities")
        return []
    
    def _empty_result_handler(self, location: str, reason: str) -> Dict:
        """Handle cases with insufficient data"""
        return {
            'success': False,
            'location': location,
            'error': reason,
            'recommendation': (
                "Try a larger radius, different location, or check data availability. "
                "Some remote areas may have limited OpenStreetMap coverage."
            )
        }


def print_analysis_results(result: Dict):
    """Pretty print analysis results"""
    print("\n" + "="*70)
    print(" HEALTHCARE ACCESSIBILITY ANALYSIS RESULTS")
    print("="*70 + "\n")
    
    if not result['success']:
        print(f"❌ Analysis Failed: {result['error']}")
        print(f"💡 Recommendation: {result['recommendation']}")
        return
    
    print(f"📍 Location: {result['location']}")
    print(f"🌐 Coordinates: {result['coordinates'][0]:.4f}, {result['coordinates'][1]:.4f}")
    print(f"🏥 Healthcare Facilities Found: {result['facilities_found']}")
    print(f"📊 Areas Analyzed: {result['areas_analyzed']}")
    
    print(f"\n🔍 Data Quality Assessment:")
    val = result['validation']
    print(f"   Status: {val['status']}")
    print(f"   Confidence: {val['confidence']}")
    print(f"   Overall Score: {val['overall_score']:.2%}")
    if val['issues']:
        print(f"   Issues:")
        for issue in val['issues']:
            print(f"      ⚠️  {issue}")
    
    print(f"\n📈 Accessibility Insights:")
    insights = result['insights']
    print(f"   Average Accessibility Score: {insights['average_score']:.1f}/100")
    print(f"   Well-Served Areas: {insights['well_served_areas']}")
    print(f"   Underserved Areas: {insights['underserved_areas']}")
    if insights['average_distance_km']:
        print(f"   Average Distance to Nearest Facility: {insights['average_distance_km']:.2f} km")
    
    print(f"\n💡 Recommendations:")
    for i, rec in enumerate(insights['recommendations'], 1):
        print(f"   {i}. {rec}")
    
    print(f"\n🗺️  Interactive Map: {result['map_file']}")
    print("\n" + "="*70 + "\n")

In [18]:

# ==============================================================================
# DEMONSTRATION SCRIPT
# ==============================================================================

if __name__ == "__main__":
    """
    Demonstration script showing the agent in action
    
    Run this to see:
    - Autonomous data discovery from OpenStreetMap
    - Multi-city analysis capability
    - Quality assessment and confidence scoring
    - Spatial reasoning and accessibility calculations
    - Interactive visualization generation
    """
    
    print("\n" + "="*70)
    print(" MODULE 3 - PRACTICAL DEMONSTRATION")
    print(" Healthcare Accessibility Intelligence System")
    print("="*70 + "\n")
    
    # Initialize agent
    print("Initializing Data-Aware Agent...")
    agent = HealthcareAccessibilityAgent()
    print("✓ Agent initialized with data source registry\n")
    
    # Example 1: Well-documented city
    print("\n" + "-"*70)
    print(" EXAMPLE 1: Sumy, Ukraine")
    print("-"*70 + "\n")
    
    result1 = agent.analyze_healthcare_accessibility(
        location="Sumy, Ukraine",
        radius_km=10
    )
    print_analysis_results(result1)
    
    # Example 2: Different city
    print("\n" + "-"*70)
    print(" EXAMPLE 2: Kyiv, Ukraine")
    print("-"*70 + "\n")
    
    result2 = agent.analyze_healthcare_accessibility(
        location="Kyiv, Ukraine",
        radius_km=10
    )
    print_analysis_results(result2)
    
    # Example 3: Smaller radius
    print("\n" + "-"*70)
    print(" EXAMPLE 3:  Kharkiv, Ukraine")
    print("-"*70 + "\n")
    
    result3 = agent.analyze_healthcare_accessibility(
        location=" Kharkiv, Ukraine",
        radius_km=10
    )
    print_analysis_results(result3)



    # result4 = agent.analyze_healthcare_accessibility(
    #     location="Edinburgh, Scotland",  # Your city here
    #     radius_km=8                       # Your radius here
    # )
    # print_analysis_results(result4)



#     'nhs_portal': {
#     'base_url': 'https://api.nhs.uk/hospitals',
#     'features': ['healthcare', 'waiting_times'],
#     'coverage': 'UK only',
#     'cost': 'free',
#     'requires_auth': False
# }

#     agent = HealthcareAccessibilityAgent()
# agent.quality_thresholds['min_facilities'] = 2
# agent.quality_thresholds['max_distance_km'] = 100

    
    
    print("\n" + "="*70)
    print(" DEMONSTRATION COMPLETE")
    print("="*70)
    print("\nKey Module 3 Concepts Demonstrated:")
    print("  ✓ Autonomous data discovery from OpenStreetMap")
    print("  ✓ Multi-source integration (OSM + Nominatim)")
    print("  ✓ Spatial reasoning (Haversine distance calculations)")
    print("  ✓ Quality assessment framework (5 dimensions)")
    print("  ✓ Validation (coordinates, completeness, coverage)")
    print("  ✓ Fallback strategies (grid generation)")
    print("  ✓ Confidence scoring (HIGH/MEDIUM/LOW)")
    print("  ✓ Interactive visualization (Folium maps)")
    print("  ✓ Decision support (recommendations, not commands)")
    print("\nCheck the generated HTML files to see the interactive maps!")
    print("="*70 + "\n")

2026-02-23 10:06:36,528 - INFO - === Starting Healthcare Accessibility Analysis ===
2026-02-23 10:06:36,529 - INFO - Location: Sumy, Ukraine
2026-02-23 10:06:36,530 - INFO - Radius: 10 km
2026-02-23 10:06:36,531 - INFO - 
[STEP 1] Geocoding location...



 MODULE 3 - PRACTICAL DEMONSTRATION
 Healthcare Accessibility Intelligence System

Initializing Data-Aware Agent...
✓ Agent initialized with data source registry


----------------------------------------------------------------------
 EXAMPLE 1: Sumy, Ukraine
----------------------------------------------------------------------



2026-02-23 10:06:36,880 - INFO - ✓ Coordinates: 50.9120, 34.8028
2026-02-23 10:06:36,895 - INFO - 
[STEP 2] Discovering healthcare facilities...
2026-02-23 10:06:36,896 - INFO -   Querying OSM Overpass API...
2026-02-23 10:06:36,897 - INFO -   Search radius: 10 km (10000 m)
2026-02-23 10:06:38,760 - INFO -   Retrieved 107 facilities from OSM
2026-02-23 10:06:38,775 - INFO - ✓ Found 107 healthcare facilities
2026-02-23 10:06:38,776 - INFO - 
[STEP 3] Discovering population centers...
2026-02-23 10:06:38,778 - INFO -   Querying OSM for residential areas...
2026-02-23 10:06:40,524 - INFO -   Retrieved 7447 residential points from OSM
2026-02-23 10:06:40,534 - INFO - ✓ Found 7447 population points
2026-02-23 10:06:40,535 - INFO - 
[STEP 4] Validating data quality...
2026-02-23 10:06:40,537 - INFO - ✓ Validation: GOOD
2026-02-23 10:06:40,537 - INFO -   Confidence: HIGH
2026-02-23 10:06:40,537 - INFO - 
[STEP 5] Calculating accessibility scores...
2026-02-23 10:06:46,391 - INFO - ✓ Analyzed 


 HEALTHCARE ACCESSIBILITY ANALYSIS RESULTS

📍 Location: Sumy, Ukraine
🌐 Coordinates: 50.9120, 34.8028
🏥 Healthcare Facilities Found: 107
📊 Areas Analyzed: 7447

🔍 Data Quality Assessment:
   Status: GOOD
   Confidence: HIGH
   Overall Score: 100.00%

📈 Accessibility Insights:
   Average Accessibility Score: 94.6/100
   Well-Served Areas: 6786
   Underserved Areas: 86
   Average Distance to Nearest Facility: 1.09 km

💡 Recommendations:

🗺️  Interactive Map: healthcare_accessibility_sumy,_ukraine.html



----------------------------------------------------------------------
 EXAMPLE 2: Kyiv, Ukraine
----------------------------------------------------------------------



2026-02-23 10:07:00,828 - INFO - ✓ Coordinates: 50.4500, 30.5241
2026-02-23 10:07:00,829 - INFO - 
[STEP 2] Discovering healthcare facilities...
2026-02-23 10:07:00,830 - INFO -   Querying OSM Overpass API...
2026-02-23 10:07:00,831 - INFO -   Search radius: 10 km (10000 m)
2026-02-23 10:07:05,974 - WARNING -   OSM query failed: 429 Client Error: Too Many Requests for url: https://overpass-api.de/api/interpreter
2026-02-23 10:07:05,976 - WARNING -   Using empty fallback for facilities
2026-02-23 10:07:05,980 - INFO - ✓ Found 0 healthcare facilities
2026-02-23 10:07:05,986 - INFO - === Starting Healthcare Accessibility Analysis ===
2026-02-23 10:07:05,987 - INFO - Location:  Kharkiv, Ukraine
2026-02-23 10:07:05,988 - INFO - Radius: 10 km
2026-02-23 10:07:05,990 - INFO - 
[STEP 1] Geocoding location...



 HEALTHCARE ACCESSIBILITY ANALYSIS RESULTS

❌ Analysis Failed: No facilities found
💡 Recommendation: Try a larger radius, different location, or check data availability. Some remote areas may have limited OpenStreetMap coverage.

----------------------------------------------------------------------
 EXAMPLE 3:  Kharkiv, Ukraine
----------------------------------------------------------------------



2026-02-23 10:07:06,447 - INFO - ✓ Coordinates: 49.9923, 36.2310
2026-02-23 10:07:06,447 - INFO - 
[STEP 2] Discovering healthcare facilities...
2026-02-23 10:07:06,447 - INFO -   Querying OSM Overpass API...
2026-02-23 10:07:06,447 - INFO -   Search radius: 10 km (10000 m)
2026-02-23 10:07:11,829 - WARNING -   OSM query failed: 429 Client Error: Too Many Requests for url: https://overpass-api.de/api/interpreter
2026-02-23 10:07:11,845 - WARNING -   Using empty fallback for facilities
2026-02-23 10:07:11,849 - INFO - ✓ Found 0 healthcare facilities



 HEALTHCARE ACCESSIBILITY ANALYSIS RESULTS

❌ Analysis Failed: No facilities found
💡 Recommendation: Try a larger radius, different location, or check data availability. Some remote areas may have limited OpenStreetMap coverage.

 DEMONSTRATION COMPLETE

Key Module 3 Concepts Demonstrated:
  ✓ Autonomous data discovery from OpenStreetMap
  ✓ Multi-source integration (OSM + Nominatim)
  ✓ Spatial reasoning (Haversine distance calculations)
  ✓ Quality assessment framework (5 dimensions)
  ✓ Validation (coordinates, completeness, coverage)
  ✓ Fallback strategies (grid generation)
  ✓ Confidence scoring (HIGH/MEDIUM/LOW)
  ✓ Interactive visualization (Folium maps)
  ✓ Decision support (recommendations, not commands)

Check the generated HTML files to see the interactive maps!

